# SarcasTone on Colab
Use a persistent Drive folder and a copy of this project. Upload your own project folder to MyDrive/SarcasTone/project. Use a T4 runtime for frozen extraction if available. Drive contains permanent resources across runtime restarts. Only upload resources you have permission to store there. Keep private voice recordings local.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess
root = Path('/content/drive/MyDrive/SarcasTone')
project = root / 'project'
os.environ['SARCASTONE_HOME'] = str(root)
os.chdir(project)
assert (project / 'app.py').exists()

## Environment
Retain Colab's installed GPU PyTorch. Install the application packages into the transient runtime. If the active scientific stack changes after installation, restart the kernel, remount Drive, and rerun the path cell.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import torch, yaml
settings = yaml.safe_load((project / 'configs/default.yaml').read_text())
settings['training']['device'] = 'auto'
config = root / 'colab.yaml'
config.write_text(yaml.safe_dump(settings))
os.environ['SARCASTONE_CONFIG'] = str(config)
print(torch.__version__, torch.cuda.is_available())
def stage(*arguments):
    subprocess.run([sys.executable, '-m', 'sarcastone', *arguments], check=True)

## Choose the resources already on Drive
For feature-only training, copy datasets/processed/manifest.json, datasets/processed/audit.json and the complete feature_cache tree under the Drive root, preserving relative paths. Skip raw extraction. For a full extraction, reuse existing permanent snapshots or explicitly enable the larger downloads below. Run this cell again after a disconnection.

In [ ]:
FEATURES_ONLY = True
if not FEATURES_ONLY:
    stage('setup')
    stage('verify')
    stage('prepare')
    stage('extract')
else:
    assert (root / 'datasets/processed/manifest.json').is_file()
    assert (root / 'feature_cache/text').is_dir()

In [ ]:
stage('train', '--run', 'colab-cv')
stage('report', '--run', 'colab-cv')

## Return to Windows
Copy results/colab-cv from Drive into your local storage results folder, including run.json, splits.json, fold folders and inference.pt. Keep local pretrained snapshots and the same feature settings. The app loads checkpoints on CPU and can select colab-cv in its sidebar. To resume training on Colab, reuse the same package versions, config and run name. If the runtime library versions changed, create a fresh run and continue using the existing cached features.